# 9.14 有機率輸出，還要怎樣訓練與校準？


假設一百次被模型報為「約八成」的判斷，只對四十次。輸出雖然是合法機率，拿這些機率做決定的程式仍會誤判風險。我們要的兩件事是：選對答案，以及讓報出的機率與沒拿來訓練的題目上的實際成敗相稱。

先用有正確答案的例子訓練。Choice 可以用[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)的交叉熵（cross entropy，CE）`-log(p_correct)`，`p_correct` 是正確答案那一項的機率；yes／no 用**二元交叉熵**，也就是只有兩個候選時的同一種代價：正確答案是 yes 時算 `-log(p_yes)`，是 no 時算 `-log(1-p_yes)`。有限等級則先把每一級寫成明確規準，再對該級做分類。例如「0＝沒有必要資訊、1＝只有任務、2＝任務及所需編號都有」是我們人工訂定的三級資訊規準。[9.11](https://birdhackor.github.io/tiny-perceptron-vlm/9.11.html)列出的 **Score** 在這裡指有次序的有限等級分布，不是任意生成一個沒有標準的滿意度小數。

若有可靠的目標分布 q，也就是每個候選該得多少機率，交叉熵可寫成 `-sum(q*log(p))`：每個候選的 `-log(p)` 先乘上 q 給的比重，再加總。當 q 是 one-hot，也就是只有正確答案那一格是1、其餘都是0，其他項乘0消失，只剩 `-log(p_correct)`，就回到上面的式子。若 q 來自教師模型的猜測，模型學到的首先是貼近教師，而不是貼近事實。

交叉熵和 Brier 分數（[9.10](https://birdhackor.github.io/tiny-perceptron-vlm/9.10.html)補充的定義：各候選機率與正確答案的0或1相減、平方再加總）都屬於 **proper scoring rule**（適當評分規則）。白話說：如果某類題目的答案真的按某個比例出現，例如七成是卡片、三成是地址，模型照實報出0.7與0.3時平均代價最低，故意報得更尖或更平都會吃虧；所以這類評分鼓勵誠實報機率。不過這只在理想條件下成立：有限資料、模型限制與標籤錯誤，都會讓「用這種代價訓練好了，機率便可靠」這個結論不一定成立。

下面用同一個正確答案檢查兩種代價的方向。向量各位置依序代表卡片、地址、其他；正確答案是第一類，寫成 one-hot 就是 `y=[1,0,0]`。Brier 採三類平方差的**總和**，不再除類別數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

y = torch.tensor([1.0, 0.0, 0.0])
for p in [torch.tensor([0.8, 0.1, 0.1]), torch.tensor([0.2, 0.7, 0.1])]:
    print("CE", float(-p[0].log()), "Brier", float(((p - y) ** 2).sum()))

`-p[0].log()`取正確答案（第一項）機率的負對數；`(p - y) ** 2`逐項相減再平方，`.sum()`加總。前者 Brier 約0.06（0.2²+0.1²+0.1²），後者約1.14（0.8²+0.7²+0.1²）；交叉熵也是前者較小，約0.2231對1.6094。這一題兩種代價都偏好前者，但它不是整體校準評估。

訓練之後還要校準。[9.10](https://birdhackor.github.io/tiny-perceptron-vlm/9.10.html)的溫度縮放，要在另一份**校準資料（calibration split）**上比較事先列出的校準溫度τ，以平均交叉熵最低者為準，再把 logits 除以選定的τ做 softmax。這會改變尖銳程度，對固定候選不改 argmax，因此不能修復選錯類別本身。

所以實際資料至少分成四份，各有一個用途：訓練資料更新參數；校準資料選溫度；**決策資料（policy split）**選行動門檻，例如機率多高才直接執行（[9.16](https://birdhackor.github.io/tiny-perceptron-vlm/9.16.html)）；測試資料最後只報凍結後的成績。第13章的 policy 指選動作的策略模型，和這份資料無關。樣本有限時可以重用資料，但要清楚記錄重用了哪一份與因此多出的不確定性；不能一邊看測試資料一邊調τ。重排候選、缺候選與新措辭也要各自報分：平均校準好，不保證每個**子群**（依來源或題型切出的一小組題目）都好。

練習：在程式的清單裡加入第三個分布`torch.tensor([1 / 3, 1 / 3, 1 / 3])`，先預測 CE 約1.0986（−log(1/3)）、Brier 約0.6667（(2/3)²+(1/3)²+(1/3)²），再執行核對。一個永遠報三類均勻的模型可能不過度自信，但能因此說它有用嗎？不能，它永遠分不出卡片和地址。還要看辨識能力與任務成本；校準與正確率回答不同問題。
